# ANGEL — laboratório gratuito de vozes (Kokoro / PT-BR)

Este notebook é apenas para **testes locais na sessão gratuita do Google Colab**. Não usa API paga, chave de serviço, assinatura nem compra de créditos. Não escolha Colab Pro nem aceite qualquer solicitação de pagamento.

Objetivo: verificar se a instalação atual do Kokoro consegue sintetizar português brasileiro e comparar as vozes disponíveis. Nenhuma voz fica aprovada automaticamente para o elenco.

## 1. Iniciar o ambiente

No Colab, mantenha o ambiente padrão gratuito. A instalação pode levar alguns minutos e baixar os modelos necessários.

In [ ]:
import sys, subprocess
from IPython.display import display, Markdown

print('Python:', sys.version)
print('Preparando as ferramentas de instalação...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

print('\nTentativa 1: instalar o pacote Kokoro publicado no PyPI...')
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'kokoro', 'soundfile'])
except subprocess.CalledProcessError:
    print('\nA instalação pelo PyPI não funcionou. Tentando o repositório oficial do Kokoro...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'git+https://github.com/hexgrad/kokoro.git', 'soundfile'])

print('\nInstalando o mecanismo de apoio para pronúncia...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y espeak-ng'], check=True)

from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
import os

print('\nBibliotecas instaladas e importadas com sucesso.')

## 2. Inicializar português brasileiro

O código de idioma `p` é o indicado pelo README oficial do Kokoro para português brasileiro. A disponibilidade real das vozes será confirmada durante a execução.

In [ ]:
pipeline = KPipeline(lang_code='p')
print('Pipeline PT-BR inicializado.')

## 3. Comparar as vozes PT-BR encontradas na documentação do projeto

A lista abaixo contém IDs candidatos documentados no ecossistema Kokoro. Como a disponibilidade pode variar entre versões/modelos, cada teste é protegido: se uma voz falhar, o notebook registra o erro e continua. Não considere nenhuma delas escolhida até ouvir e avaliar.

In [ ]:
voice_candidates = ['pf_dora', 'pm_alex', 'pm_santa']
sample_text = ('Porque conseguir sozinho não significa que alguém precise fazer tudo sozinho. ' 
               'Você demorou. Eu sei.')
os.makedirs('angel_voice_tests', exist_ok=True)
successful_voices = []

for voice_id in voice_candidates:
    print(f'\n--- Testando {voice_id} ---')
    try:
        chunks = list(pipeline(sample_text, voice=voice_id, speed=1.0))
        if not chunks:
            print('Sem áudio retornado.')
            continue
        audio_parts = [chunk[2] for chunk in chunks]
        import numpy as np
        audio = np.concatenate(audio_parts) if len(audio_parts) > 1 else audio_parts[0]
        output_path = f'angel_voice_tests/{voice_id}.wav'
        sf.write(output_path, audio, 24000)
        display(Audio(data=audio, rate=24000))
        print(f'Arquivo salvo: {output_path}')
        successful_voices.append(voice_id)
    except Exception as exc:
        print(f'Não foi possível testar {voice_id}: {type(exc).__name__}: {exc}')

print('\nVozes que geraram áudio:', successful_voices)

## 4. Baixar os testes

Execute esta célula depois dos testes. Ela cria um ZIP com os WAVs que foram gerados com sucesso.

In [ ]:
import zipfile
from pathlib import Path

wav_files = list(Path('angel_voice_tests').glob('*.wav'))
if not wav_files:
    print('Nenhum WAV foi gerado. Confira as mensagens da etapa anterior.')
else:
    with zipfile.ZipFile('ANGEL_testes_de_voz_Kokoro.zip', 'w') as archive:
        for file_path in wav_files:
            archive.write(file_path, arcname=file_path.name)
    from google.colab import files
    files.download('ANGEL_testes_de_voz_Kokoro.zip')

## Critérios de avaliação para o elenco ANGEL

- Pronúncia natural em português brasileiro, sem sotaque artificial indesejado.
- Cadência, pausas e respiração convincentes.
- Capacidade de transmitir intenção sem soar excessivamente robótica.
- Consistência entre execuções.
- Licença e condições de uso adequadas para a distribuição futura do anime.

**Limites:** o Colab gratuito pode impor cotas, encerrar sessões e variar a disponibilidade de hardware. O notebook não garante qualidade de dublagem profissional. Não envie gravações pessoais nem tente clonar a voz de pessoas reais. Os testes não autorizam uso comercial automático: a licença do modelo, dos pesos e das vozes precisa ser conferida antes de qualquer lançamento.